# Aula 01 — Processamento de Linguagem Natural

Células na ordem dos slides; algumas dependem de variáveis de células anteriores.

## Slide 30 — POLISSEMIA EM CÓDIGO

*Bloco 01*

In [ ]:
# A palavra "banco" tem múltiplos sentidos em português
from sklearn.feature_extraction.text import CountVectorizer
corpus = [
    "O banco aprovou meu empréstimo ontem",       # financeiro
    "Sentei no banco da praça para descansar",     # móvel
    "O banco de dados PostgreSQL caiu novamente",  # tech
]
vec = CountVectorizer()
X = vec.fit_transform(corpus)
print('Vocabulário:', vec.get_feature_names_out())
# Para o BoW, 'banco' aparece igual nos 3 documentos.
# Não há nada que distinga sentido financeiro, móvel ou tech!
print('Matriz documento-termo:')
print(X.toarray())
# A coluna de 'banco' tem 1 em todas as linhas → polissemia perdida

## Slide 49 — LEI DE ZIPF A DISTRIBUIÇÃO ESTATÍSTICA DOS TOKENS EM QUALQUER CORPUS

*Bloco 02*

In [ ]:
# Lei de Zipf na prática (em português)
# Token mais frequente em corpus de português:
# 'de' (~5% de TODOS os tokens)
# 'a' (~3%)
# 'o' (~3%)
# 'que' (~2%)
# ...
# Palavras como 'arqueólogo': aparecem 1-2 vezes
# 50% do vocabulário aparece APENAS UMA VEZ (hapax legomena)
# Implicação prática:
# Top 100 palavras = ~50% de todos os tokens do corpus
# Top 1000 palavras = ~80%
# Resto do vocabulário (milhares de palavras) = ~20%

## Slide 52 — SETUP DO AMBIENTE BIBLIOTECAS PARA COMEÇAR

*Bloco 03*

In [ ]:
# Instalar bibliotecas
# pip install nltk spacy scikit-learn
# python -m spacy download pt_core_news_lg
import nltk
import spacy
import pandas as pd
from collections import Counter
# Recursos do NLTK em português
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('rslp')          # stemmer português
# Carregar modelo do spaCy em português
nlp = spacy.load('pt_core_news_lg')
# Corpus exemplo: reviews fictícias de produtos
corpus = [
    "O produto é excelente, recomendo muito! Chegou rápido.",
    "Péssimo, o aparelho veio com defeito. Não comprem.",
    "Ótima qualidade pelo preço. Entrega rápida e bem embalado.",
    "Adorei! Funciona perfeitamente, superou minhas expectativas.",
    "Esperei 3 horas no telefone. Adorei o atendimento.",  # ironia
]

## Slide 53 — PASSO 1 — CARREGANDO O CORPUS COM NLTK TOKENIZAÇÃO E CONTAGEM

*Bloco 04*

In [ ]:
from nltk.tokenize import word_tokenize
# Tokenizar todos os documentos
todos_tokens = []
for doc in corpus:
    tokens = word_tokenize(doc.lower(), language='portuguese')
    todos_tokens.extend(tokens)
print(f'Total de tokens: {len(todos_tokens)}')
print(f'Vocabulário (únicos): {len(set(todos_tokens))}')
# Top 10 palavras mais frequentes (Lei de Zipf)
contador = Counter(todos_tokens)
print('\nTop 10 mais frequentes:')
for palavra, freq in contador.most_common(10):
    print(f'  {palavra:15s} → {freq}')
# Saída esperada:
# o, ., e, , o, com, a (stop words dominam)

## Slide 54 — PASSO 2 — ANÁLISE LINGUÍSTICA COM SPACY O PADRÃO DA INDÚSTRIA

*Bloco 05*

In [ ]:
# spaCy oferece tokenização robusta + POS tagging + lemmatization + NER
doc = nlp("O João comprou um livro na Amazon ontem.")
print(f"{'TOKEN':12s} {'LEMA':12s} {'POS':10s} {'DEP':10s}")
print('-' * 50)
for token in doc:
    print(f"{token.text:12s} {token.lemma_:12s} "
          f"{token.pos_:10s} {token.dep_:10s}")
# Saída esperada:
# TOKEN        LEMA         POS        DEP
# O            o            DET        det
# João         João         PROPN      nsubj
# comprou      comprar      VERB       ROOT
# um           um           DET        det
# livro        livro        NOUN       obj
# na           em o         ADP        case
# Amazon       Amazon       PROPN      obl
# ontem        ontem        ADV        advmod
# .            .            PUNCT      punct

## Slide 55 — SPACY: NAMED ENTITY RECOGNITION IDENTIFICANDO PESSOAS, LUGARES E ORGANIZAÇÕES

*Bloco 06*

In [ ]:
# Identificação de entidades nomeadas em português
texto = ("Leonardo da Vinci nasceu em Vinci, Itália, em 1452. "
        "Foi pintor, cientista e trabalhou para Lorenzo de Médici. "
        "Morreu em Amboise, na França.")
doc = nlp(texto)
print('Entidades encontradas:')
for ent in doc.ents:
    print(f'  {ent.text:25s} → {ent.label_}')
# Saída esperada:
# Leonardo da Vinci         → PER  (pessoa)
# Vinci                     → LOC  (local)
# Itália                    → LOC
# 1452                      → MISC
# Lorenzo de Médici         → PER
# Amboise                   → LOC
# França                    → LOC

## Slide 56 — PASSO 3 — PRIMEIRA VETORIZAÇÃO DO TEXTO PRÉ-PROCESSADO A VETORES NUMÉRICOS

*Bloco 07*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
stopwords_pt = stopwords.words('portuguese')
# Bag of Words com remoção de stop words
vec = CountVectorizer(stop_words=stopwords_pt)
X = vec.fit_transform(corpus)
print(f'Shape da matriz: {X.shape}')  # (5 docs, N features)
print(f'Vocabulário ({len(vec.vocabulary_)} termos):')
print(vec.get_feature_names_out()[:15])
# Visualizar matriz documento-termo (esparsa!)
import pandas as pd
df_dt = pd.DataFrame(
    X.toarray(),
    columns=vec.get_feature_names_out()
)
print('\nMatriz documento-termo (primeiras colunas):')
print(df_dt.iloc[:, :8])